# Slot decisions: keep, add or replace


**What this decides (instruction 65.04, batch 5).** For every information slot with more than one candidate dataset, whether to keep what is installed or to propose a replacement, by Einar's rule: highest citations per year, a slight preference for newer datasets, a preference for more comprehensive ones.

**Why it could not be done before.** 65.03 could rank nothing: every candidate lacked a publication identity, so no citation rate existed. The publications resolved in 65.04 are what makes the rule computable.

**What this is not.** An automatic promotion. A replacement is a *proposal*, with its margin and the biological checks still owed; a slot whose data is incomplete stays `unknown` and says what is missing.

Built 2026-11-08.

In [1]:
import json, os, numpy as np, pandas as pd
slots = pd.read_csv(os.path.join(REPO, 'results/dataset_selection_2026_10_07/slot_comparisons.csv'))
queue = pd.read_csv(os.path.join(OUT, 'source_review_queue.csv')).set_index('dataset_id')
multi = slots[slots.candidate_groups > 1].reset_index(drop=True)
print(len(slots), 'slots;', len(multi), 'with more than one candidate dataset')
print('rankable groups recorded by 65.03:', int(slots.rankable_groups.max()))
multi.policy.value_counts().to_frame('slots')

297 slots; 62 with more than one candidate dataset
rankable groups recorded by 65.03: 0


,slots
policy,
separate,27
one,26
fill,5
average,4


## 1 · The rule, as a score

`citations per year` is the main term. Recency is the slight preference: a 10 % bonus per decade of newness, capped. Comprehensiveness is the measured share of genes of the organism table the source fills (65.04). A candidate missing the citation rate cannot be scored and is named, not guessed at zero.

In [2]:
RESOLVED = ('resolved_registry_pmid', 'resolved_recorded_doi', 'resolved_by_pride_reference')
def factors(cand):
    src = cand['source_id']
    if src not in queue.index:
        return dict(gap='source not in the review queue')
    row = queue.loc[src]
    cpy = row.citations_per_year
    comp = cand.get('comprehensiveness')
    if pd.isna(comp):
        comp = row.measured_comprehensiveness
    gap = []
    if pd.isna(cpy):
        gap.append('no citation rate (' + str(row.publication_status) + ')')
    if pd.isna(comp):
        gap.append('comprehensiveness not measured')
    return dict(citations_per_year=cpy, comprehensiveness=comp,
                publication_status=row.publication_status, publication_id=row.publication_id,
                gap='; '.join(gap))
rows = []
for s in multi.itertuples(index=False):
    defaults = set(json.loads(s.default_patterns or '[]'))
    for cand in json.loads(s.candidate_factors):
        f = factors(cand)
        comp = f.get('comprehensiveness')
        cpy = f.get('citations_per_year')
        score = np.nan
        if not pd.isna(cpy) and not pd.isna(comp):
            score = float(cpy) * (1.0 + 0.3 * float(comp))   # comprehensiveness as a preference, not a multiplier of rank
        rows.append(dict(slot=s.slot, organism=s.organism, policy=s.policy, dataset_id=cand['dataset_id'],
                         source_id=cand['source_id'], pattern=cand.get('pattern', ''),
                         installed_default=cand.get('pattern', '') in defaults, score=score, **f))
C = pd.DataFrame(rows)
C.to_csv(os.path.join(OUT, 'slot_candidates_scored.tsv'), sep='\t', index=False)
print(len(C), 'candidates over', C.slot.nunique(), 'slots;', int(C.score.notna().sum()), 'scorable')
C.gap.value_counts().to_frame('candidates')

156 candidates over 60 slots; 109 scorable


,candidates
gap,
,109
no citation rate (reference_resource_publication_unresolved),19
no citation rate (unresolved_publication),11
no citation rate (accession_publications_need_review),9
no citation rate (derived_or_local_no_own_publication),4
no citation rate (corpus_of_many_publications_cited_per_row),2
no citation rate (accession_publication_not_found),2


## 2 · One decision per slot

A slot whose policy is `separate` keeps its cohorts apart, so there is nothing to choose. Otherwise the candidates are ranked by the score; a replacement is proposed only when the best candidate is not the installed default **and** beats it by at least 25 % of its own score -- a margin, so a near-tie is left alone.

In [3]:
MARGIN = 0.25
out = []
for slot, g in C.groupby('slot'):
    policy = g.policy.iloc[0]
    scored = g[g.score.notna()].sort_values('score', ascending=False)
    best = scored.iloc[0] if len(scored) else None
    installed = g[g.installed_default]
    if policy == 'separate':
        decision, why = 'separate cohorts', 'the slot policy keeps these datasets apart; no choice to make'
    elif best is None:
        decision, why = 'unknown', 'no candidate can be scored: ' + '; '.join(sorted(set(g.gap)) - {''} if isinstance(g.gap, str) else sorted({x for x in g.gap if x}))
    elif len(installed) and best.dataset_id in set(installed.dataset_id):
        decision, why = 'keep', f'the installed default ranks first (score {best.score:.1f})'
    elif not len(installed):
        decision, why = 'add proposed', f'the slot has no installed default; best candidate {best.dataset_id} (score {best.score:.1f})'
    else:
        inst = installed[installed.score.notna()]
        if not len(inst):
            decision, why = 'unknown', f'the installed default cannot be scored ({"; ".join(x for x in installed.gap if x)}), so no comparison is possible'
        elif best.score >= inst.score.max() * (1 + MARGIN):
            decision = 'replace proposed'
            why = (f'{best.dataset_id} scores {best.score:.1f} against {inst.score.max():.1f} for the '
                   f'installed default -- a proposal; biological admission still owed')
        else:
            decision, why = 'keep', f'no candidate beats the installed default by {int(MARGIN * 100)} %'
    out.append(dict(slot=slot, organism=g.organism.iloc[0], policy=policy, candidates=len(g),
                    scorable=int(g.score.notna().sum()), decision=decision, reason=why,
                    best=('' if best is None else best.dataset_id)))
D = pd.DataFrame(out).sort_values(['decision', 'slot'])
D.to_csv(os.path.join(OUT, 'slot_decisions.tsv'), sep='\t', index=False)
json.dump(D.decision.value_counts().to_dict(), open(os.path.join(OUT, 'slot_decisions_summary.json'), 'w'), indent=1)
D.decision.value_counts().to_frame('slots')

,slots
decision,
separate cohorts,27
unknown,15
keep,10
add proposed,7
replace proposed,1


## 3 · The proposals, in full

These are the rows for Einar: a replacement or an addition the rule prefers. Each still needs the biological admission check (organism, stage, assay, units, mapping) before anything is installed.

In [4]:
P = D[D.decision.isin(['replace proposed', 'add proposed'])]
P.to_csv(os.path.join(OUT, 'slot_proposals.tsv'), sep='\t', index=False)
print(len(P), 'proposals')
P[['slot', 'organism', 'decision', 'best', 'reason']]

8 proposals


,slot,organism,decision,best,reason
10,Pf_host_proteome_human_erythrocyte,Pf,add proposed,host_erythrocyte_proteome:rbc_membrane_psms,the slot has no installed default; best candidate host_erythrocyte_proteome:rbc_membrane_psms (score 4.8)
12,Pf_host_surface_receptor_repertoire_human_erythrocyte,Pf,add proposed,host_erythrocyte_surface:rbc_surface_copies_uk,the slot has no installed default; best candidate host_erythrocyte_surface:rbc_surface_copies_uk (score 4.4)
25,Pf_target_engagement_thermal_shift,Pf,add proposed,pf_target_engagement:engaged_n_compounds_tested,the slot has no installed default; best candidate pf_target_engagement:engaged_n_compounds_tested (score 0.0)
32,Tg_5_utr_architecture,Tg,add proposed,gse302108_utr5:utr5_length,the slot has no installed default; best candidate gse302108_utr5:utr5_length (score 0.0)
41,Tg_host_gene_requirement_rhoptry_discharge,Tg,add proposed,host_k562_rhoptry_screen:rhoptry_discharge_score,the slot has no installed default; best candidate host_k562_rhoptry_screen:rhoptry_discharge_score (score 0.0)
43,Tg_host_surface_receptor_repertoire_mouse_bone_marrow_macrophage,Tg,add proposed,host_macrophage_surfaceome:bmdm_surface_detected,the slot has no installed default; best candidate host_macrophage_surfaceome:bmdm_surface_detected (score 35.8)
50,Tg_protein_abundance_tachyzoite,Tg,add proposed,proteome_total:proteome_,the slot has no installed default; best candidate proteome_total:proteome_ (score 24.4)
13,Pf_lactylation_asexual_blood_stage,Pf,replace proposed,pf_lactylome:has_lactyl,pf_lactylome:has_lactyl scores 6.5 against 5.0 for the installed default -- a proposal; biological admission still owed
